# 🧪 Prompt Format Ablation Tester
**Does Prompt Structure Affect LLM Price Prediction? A Controlled Experiment**

_Week 7 Community Exercise · LiteLLM · HuggingFace · Gradio_

---

## What I Learned in Week 7

Week 7 was all about fine-tuning open-source models. A key finding from Day 2 was how dramatically **prompt format** shapes what the model learns and how it responds:
- **Day 2** — `item.make_prompts(tokenizer, CUTOFF, include_label)` shows that the exact wording, token count, and structure of prompts matters hugely
- **Day 5** — Evaluation: the same model weights can produce wildly different MAE scores depending on how you ask the question

## Why This Project

I wanted to *prove* this empirically: take the **same model**, run the **same 20 test items**, using **3 different prompt formats**, and measure the accuracy difference. Because if format matters for fine-tuning (Day 2), it also matters for inference.

The headline feature: a **"Try Your Own Prompt"** tab where you write any template and instantly get a live accuracy score.

---

## Components

| Component | Purpose |
|---|---|
| `ed-donner/items_lite` (HuggingFace Hub) | Pre-processed Amazon product dataset |
| `litellm` + Ollama (or any provider) | Run the same model across all prompt formats |
| 3 built-in prompt formats | Minimal, Structured, Chain-of-Thought |
| `gr.Blocks`, `gr.BarPlot`, `gr.Textbox` | Interactive comparison + live custom prompt tester |

---

## How It Works

1. **Load** — Pull `ed-donner/items_lite` test split from HuggingFace Hub
2. **Define formats** — 3 built-in prompt templates (Minimal, Structured, Chain-of-Thought)
3. **Benchmark** — Run all 3 formats on the same 20 items, collect MAE per format
4. **Explore** — Gradio UI: bar chart comparing formats + custom prompt live tester

---

## Model Configuration

| Setting | Default | Alternative |
|---|---|---|
| `MODEL_ID` | `ollama/llama3.2` | Any HF model ID, `openai/gpt-4.1-nano`, etc. |
| `BENCHMARK_SIZE` | `20` | Lower = faster, higher = more accurate results |
| API requirement | None (Ollama) | Needs key if using OpenAI/Anthropic/OpenRouter |

**Hardware:** CPU-only locally via Ollama. Point `MODEL_ID` at a HuggingFace fine-tuned model to compare your own Week 7 output!

## Cell 1 — Install Dependencies

In [ ]:
# Run once — restart kernel after installation if needed
!uv pip install -q litellm gradio datasets python-dotenv

print("✅ Dependencies installed.")

## Cell 2 — Imports & Configuration

In [ ]:
import os
import re
import random
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv(override=True)

# ── API / Model config ───────────────────────────────────────────────────────
#
#  Default: Ollama running locally (free, no API key)
#  To use a fine-tuned HF model from Week 7, change MODEL_ID to:
#      "huggingface/<your-username>/<your-model>"
#  Or switch to a frontier model:
#      "openai/gpt-4.1-nano"  →  requires OPENAI_API_KEY in .env
#      "openrouter/openai/gpt-4.1-nano"  →  requires OPENROUTER_API_KEY
#
MODEL_ID       = "ollama/llama3.2"     # ← change this to your model
OLLAMA_BASE    = "http://localhost:11434"  # only used when MODEL_ID starts with "ollama/"

BENCHMARK_SIZE = 20    # items to evaluate per format
RANDOM_SEED    = 42
random.seed(RANDOM_SEED)

# HuggingFace login (needed for dataset loading)
hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(hf_token, add_to_git_credential=True)
    print(f"HuggingFace token: {hf_token[:8]}...")
else:
    print("⚠️  No HF_TOKEN — dataset loading will attempt anonymous access")

print(f"Model: {MODEL_ID}")
print(f"Benchmark size: {BENCHMARK_SIZE} items per format")
print("\n✅ Configuration ready.")

## Cell 3 — Load Dataset from HuggingFace Hub

In [ ]:
from datasets import load_dataset

print("Loading ed-donner/items_lite from HuggingFace Hub...")

raw = load_dataset("ed-donner/items_lite")

test_data = [
    {"summary": r["summary"], "price": float(r["price"])}
    for r in raw["test"]
]

print(f"Test items loaded: {len(test_data):,}")
print(f"\nExample item:")
print(f"  Summary: {test_data[0]['summary'][:150]}...")
print(f"  Price  : ${test_data[0]['price']:.2f}")

# Fix benchmark subset
random.seed(RANDOM_SEED)
bench_items = random.sample(test_data, min(BENCHMARK_SIZE, len(test_data)))
print(f"\nBenchmark subset: {len(bench_items)} items (fixed seed={RANDOM_SEED})")

## Cell 4 — Define 3 Prompt Formats

In [ ]:
# ── Format A: Minimal ───────────────────────────────────────────────────────
# One-liner instruction. As close to the raw text as possible.
# Mirrors how many fine-tuning datasets look when no system prompt is used.

PROMPT_A_SYSTEM = "Price this product. Reply with the dollar amount only."

def format_a(item):
    return PROMPT_A_SYSTEM, item["summary"]


# ── Format B: Structured ────────────────────────────────────────────────────
# Mirrors the Week 7 Day 2 prompt pattern: explicit role, rules, and output format.

PROMPT_B_SYSTEM = """You are a professional retail pricing analyst.
Your task is to estimate the retail price of a product based on its description.

Rules:
- Consider the product category, brand, and features
- Price must be between $1 and $1000
- Respond ONLY with the price in the format: $XX.XX
- Do NOT include any explanation"""

def format_b(item):
    user_msg = f"Product description:\n{item['summary']}\n\nEstimated price:"
    return PROMPT_B_SYSTEM, user_msg


# ── Format C: Chain-of-Thought (compressed) ─────────────────────────────────
# Asks the model to reason briefly before giving the price.
# Inspired by Week 7 Day 5 observation that reasoning prompts can improve accuracy.

PROMPT_C_SYSTEM = """You are a pricing expert. Think step by step:
1. Identify the product type and typical price range
2. Consider brand, quality signals, and features mentioned
3. Arrive at a specific price estimate

Format your response as:
Reasoning: <1-2 sentences only>
Price: $XX.XX"""

def format_c(item):
    return PROMPT_C_SYSTEM, item["summary"]


FORMATS = {
    "A — Minimal"             : format_a,
    "B — Structured"          : format_b,
    "C — Chain-of-Thought"    : format_c,
}

# Preview each format
for name, fn in FORMATS.items():
    sys_p, usr_p = fn(bench_items[0])
    print(f"\n{'='*60}")
    print(f"Format {name}")
    print(f"  System: {sys_p[:80]}...")
    print(f"  User  : {usr_p[:80]}...")

print("\n✅ 3 prompt formats defined.")

## Cell 5 — Run the Ablation Benchmark

In [ ]:
from litellm import completion

def extract_price(text: str) -> float:
    """Extract a numeric price from any LLM reply."""
    text = str(text).replace(",", "").replace("$", "")
    # Look for 'Price: X' pattern first (for CoT format)
    price_match = re.search(r"[Pp]rice[:\s]+([\d.]+)", text)
    if price_match:
        return float(price_match.group(1))
    # Fall back to first number
    match = re.search(r"[-+]?\d*\.?\d+", text)
    return float(match.group()) if match else 0.0

def call_model(system_prompt, user_prompt):
    """Call the configured model with given prompts."""
    kwargs = dict(
        model=MODEL_ID,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        max_tokens=80,
    )
    if MODEL_ID.startswith("ollama/"):
        kwargs["api_base"] = OLLAMA_BASE
    return completion(**kwargs).choices[0].message.content.strip()

def benchmark_format(format_fn, label, items):
    """Run format_fn over all items, return MAE."""
    errors = []
    for item in items:
        try:
            sys_p, usr_p = format_fn(item)
            reply  = call_model(sys_p, usr_p)
            pred   = extract_price(reply)
            errors.append(abs(pred - item["price"]))
        except Exception as e:
            print(f"  [{label}] Error: {e}")
    mae = round(sum(errors) / len(errors), 2) if errors else 9999.0
    return mae

# Run benchmark
ablation_results = {}

for name, fn in FORMATS.items():
    print(f"Benchmarking format: {name} ...")
    mae = benchmark_format(fn, name, bench_items)
    ablation_results[name] = mae
    print(f"  → MAE = ${mae:.2f}")

best   = min(ablation_results, key=ablation_results.get)
worst  = max(ablation_results, key=ablation_results.get)
spread = ablation_results[worst] - ablation_results[best]

print(f"\n✅ Ablation complete.")
print(f"   Best format:  {best} (${ablation_results[best]:.2f})")
print(f"   Worst format: {worst} (${ablation_results[worst]:.2f})")
print(f"   Spread: ${spread:.2f} — that's the cost of prompt choice!")

## Cell 6 — Gradio UI: Results + "Try Your Own Prompt"

In [ ]:
import pandas as pd
import gradio as gr

def build_results_df():
    sorted_results = sorted(ablation_results.items(), key=lambda x: x[1])
    best_mae = sorted_results[0][1]
    rows = []
    for rank, (name, mae) in enumerate(sorted_results, start=1):
        rows.append({
            "Rank"    : rank,
            "Format"  : name,
            "MAE ($)" : f"{mae:.2f}",
            "vs Best" : "🏆 Best" if rank == 1 else f"+${mae - best_mae:.2f}",
        })
    return pd.DataFrame(rows)

def build_chart_df():
    return pd.DataFrame([
        {"Format": name, "MAE": mae}
        for name, mae in sorted(ablation_results.items(), key=lambda x: x[1])
    ])

def run_custom_prompt(custom_system: str, custom_user_template: str, progress=gr.Progress()):
    """
    Let user supply their own system prompt and user template.
    Template can use {product} as a placeholder for the product summary.
    """
    if not custom_system.strip():
        return "⚠️ Please enter a system prompt.", None
    if not custom_user_template.strip():
        return "⚠️ Please enter a user message template (use {product} for the product text).", None

    errors = []
    progress(0, desc="Running custom prompt...")

    for i, item in enumerate(bench_items):
        try:
            user_msg = custom_user_template.replace("{product}", item["summary"])
            reply    = call_model(custom_system, user_msg)
            pred     = extract_price(reply)
            errors.append(abs(pred - item["price"]))
        except Exception as e:
            errors.append(None)
        progress((i + 1) / len(bench_items), desc=f"Item {i+1}/{len(bench_items)}")

    errors = [e for e in errors if e is not None]
    if not errors:
        return "❌ All items failed — check your model config.", None

    custom_mae = round(sum(errors) / len(errors), 2)
    best_mae   = min(ablation_results.values())
    diff       = custom_mae - best_mae
    verdict    = "🏆 New best!" if diff < 0 else (f"+${diff:.2f} vs best format" if diff > 0 else "Tied with best!")

    summary = (
        f"**Your prompt MAE: ${custom_mae:.2f}**  ·  {verdict}\n\n"
        f"| Format | MAE |\n|---|---|\n"
    )
    for name, mae in sorted(ablation_results.items(), key=lambda x: x[1]):
        summary += f"| {name} | ${mae:.2f} |\n"
    summary += f"| **Your prompt** | **${custom_mae:.2f}** |\n"

    # Updated chart with custom prompt
    all_results = dict(ablation_results)
    all_results["Your Prompt"] = custom_mae
    chart = pd.DataFrame([
        {"Format": n, "MAE": m}
        for n, m in sorted(all_results.items(), key=lambda x: x[1])
    ])
    return summary, chart


# ── Build the Gradio app ─────────────────────────────────────────────────────
with gr.Blocks(title="Prompt Format Ablation Tester") as demo:

    gr.Markdown(
        "# 🧪 Prompt Format Ablation Tester\n"
        "**How much does prompt wording affect price prediction accuracy?**\n\n"
        f"_Model: `{MODEL_ID}` · Benchmark: {len(bench_items)} Amazon products · Metric: MAE_"
    )

    with gr.Tabs():

        with gr.Tab("📊 Ablation Results"):
            gr.Markdown("### The 3 built-in formats, ranked by MAE (lower = better)")
            gr.Dataframe(value=build_results_df(), interactive=False)
            gr.Markdown("### Visual comparison")
            gr.BarPlot(
                value=build_chart_df(),
                x="Format",
                y="MAE",
                title="MAE by Prompt Format — Lower is Better",
                y_title="MAE ($)",
                height=350,
            )

        with gr.Tab("✍️ Try Your Own Prompt"):
            gr.Markdown(
                "### Write your own prompt and benchmark it against the 3 built-in formats\n"
                "`{product}` in your user message will be replaced with the product description."
            )
            with gr.Row():
                custom_system = gr.Textbox(
                    label="System Prompt",
                    value="You are an expert pricing assistant. Estimate the product price and reply with only the dollar amount.",
                    lines=4,
                )
                custom_user = gr.Textbox(
                    label="User Message Template  (use {product} for the product text)",
                    value="Product: {product}\n\nPrice estimate:",
                    lines=4,
                )
            run_btn = gr.Button("🚀 Run Benchmark", variant="primary", size="lg")

            result_md  = gr.Markdown()
            result_chart = gr.BarPlot(
                x="Format",
                y="MAE",
                title="Your Prompt vs Built-in Formats",
                y_title="MAE ($)",
                height=350,
            )

            gr.Examples(
                examples=[
                    ["Respond with a single dollar price. Nothing else.",  "Estimate price: {product}"],
                    ["You are shopping on Amazon. What would you pay for this?", "Item: {product}\nMy budget estimate:"],
                    ["Think like a wholesaler. What's the manufacturer's suggested retail price?", "Product: {product}"],
                ],
                inputs=[custom_system, custom_user],
                label="Example templates to try",
            )

            run_btn.click(
                run_custom_prompt,
                inputs=[custom_system, custom_user],
                outputs=[result_md, result_chart],
            )

demo.launch(inbrowser=True)

---

## Concepts Demonstrated

| Cell | Concept | What it shows |
|---|---|---|
| 3 | HF `datasets` | Load pre-processed dataset from Hub in one line |
| 4 | Prompt engineering | System prompt role, rules, output format — mirrors Week 7 Day 2 `make_prompts()` |
| 5 | Ablation study | Hold model/data constant; vary only the prompt — controlled experiment |
| 5 | Price extraction | Parsing LLM string output including Chain-of-Thought format |
| 6 | Gradio live testing | Interactive custom prompt sandbox with `gr.Progress` feedback |
| 6 | `gr.Examples` | Clickable example templates — mirrors Week 4 Gradio pattern |

---

## Extensions Possible

1. **Swap in your own fine-tuned model** — set `MODEL_ID` to your HF Hub model ID from Week 7
2. **Compare base vs fine-tuned** — run the same 3 formats on both and see if fine-tuning reduces sensitivity to prompt choice
3. **Token efficiency** — track token count per format (shorter format + better accuracy = ideal)
4. **Category breakdown** — filter bench items by category and see if prompt sensitivity varies across product types
5. **Temperature ablation** — add `temperature` as a slider in the UI and see how it interacts with format choice
